In [51]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_chroma import Chroma

from dotenv import load_dotenv

import pymupdf
import os

In [56]:
def vector_store_for_storing_embeddings(chunks):

    embeddings = create_embeddings()

    vector_store = Chroma(
        collection_name="rag_fusion_test",
        embedding_function=embeddings,
        persist_directory="./vector_store"
    )

    batch_size = 5

    for i in range(0, len(chunks), batch_size):

        batch = chunks[i:i + batch_size]

        vector_store.add_documents(batch)

        print(f"Processed {min(i + batch_size, len(chunks))}/{len(chunks)}")

    return vector_store

In [57]:
PDF_PATH = r"C:\Users\HP\Downloads\Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf"

documents = load_documents(PDF_PATH)

chunks = split_documents(documents)

vector_store = vector_store_for_storing_embeddings(chunks)

length of pdf file: 392
Processed 5/973
Processed 10/973
Processed 15/973
Processed 20/973
Processed 25/973
Processed 30/973
Processed 35/973
Processed 40/973
Processed 45/973
Processed 50/973
Processed 55/973
Processed 60/973
Processed 65/973
Processed 70/973
Processed 75/973
Processed 80/973
Processed 85/973
Processed 90/973
Processed 95/973
Processed 100/973
Processed 105/973
Processed 110/973
Processed 115/973
Processed 120/973
Processed 125/973
Processed 130/973
Processed 135/973
Processed 140/973
Processed 145/973
Processed 150/973
Processed 155/973
Processed 160/973
Processed 165/973
Processed 170/973
Processed 175/973
Processed 180/973
Processed 185/973
Processed 190/973
Processed 195/973
Processed 200/973
Processed 205/973
Processed 210/973
Processed 215/973
Processed 220/973
Processed 225/973
Processed 230/973
Processed 235/973
Processed 240/973
Processed 245/973
Processed 250/973
Processed 255/973
Processed 260/973
Processed 265/973
Processed 270/973
Processed 275/973
Proces

In [ ]:
def load_documents(pdf_path):
    loader=PyMuPDFLoader(pdf_path)
    documents=loader.load()
    print("length of pdf file:",len(documents))
    return documents

def split_documents(documents):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
    chunks=text_splitter.split_documents(documents)
    
    return chunks
from langchain_ollama import OllamaEmbeddings

def create_embeddings():

    embeddings = OllamaEmbeddings(
        model="nomic-embed-text"
    )

    return embeddings

from langchain_chroma import Chroma


def vector_store_for_storing_embeddings(chunks):
    embeddings=create_embeddings()
    
    vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./vector_store",
    collection_name="rag_fusion")

    return vector_store

from langchain_ollama import ChatOllama
llm=ChatOllama(model="llama3.2:3b",temperature=0.7)

query_prompt=ChatPromptTemplate.from_template(""" 
you are a powerful ai assistant helping the rag system retive revelent information.


Your task:
Generate exactly 5 alternative search queries for the user's original question.

Rules:
1. Preserve the original meaning and intent.
2. Use different wording and perspectives.
3. Make each query useful for retrieving relevant documents.
4. Do not answer the questions.
5. Do not add explanations or introductory text.

Output format:
Return only a numbered list:
1. ...
2. ...
3. ...
4. ...
5. ...

Original question:
{question}
""")



chain=query_prompt|llm

def generate_queries(question):
    response=chain.invoke({"question":question})

    queries = response.content.strip().split("\n")

    cleaned_queries = []

    for query in queries:
        query=query.strip()
        if query:
            query=query.split(".",1)[-1]
            cleaned_queries.append(query)

    return cleaned_queries



def reterving_answers_for_each_multiple_queries(queries):

    all_results = []

    for query in queries:

        result = vector_store.similarity_search(
            query,
            k=3
        )

        all_results.append(result)

    return all_results


#MULTI QUERY FUSION
def reciprocal_rank_fusion(all_results, k=60):

    rrf_scores = {}
    documents = {}

    for results in all_results:

        for rank, doc in enumerate(results, start=1):

            doc_id = (
                doc.metadata.get("source"),
                doc.metadata.get("page"),
                doc.page_content
            )

            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0
                documents[doc_id] = doc

            rrf_scores[doc_id] += 1 / (k + rank)

    ranked_documents = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        (documents[doc_id], score)
        for doc_id, score in ranked_documents
    ] 

In [ ]:
embeddings = create_embeddings()

test_vector = embeddings.embed_query(
    "What is machine learning?"
)

print("Embedding dimensions:", len(test_vector))
print("Embedding test successful!")

Embedding dimensions: 768
Embedding test successful!


In [ ]:
embeddings = create_embeddings()

test_texts = [
    chunk.page_content
    for chunk in chunks[:5]
]

test_vectors = embeddings.embed_documents(test_texts)

print("Number of embeddings:", len(test_vectors))
print("Dimensions:", len(test_vectors[0]))

Number of embeddings: 5
Dimensions: 768


In [ ]:
from langchain_chroma import Chroma

def vector_store_for_storing_embeddings(chunks):

    embeddings = create_embeddings()

    vector_store = Chroma(
        collection_name="rag_fusion",
        embedding_function=embeddings,
        persist_directory="./vector_store"
    )

    batch_size = 20

    for i in range(0, len(chunks), batch_size):

        batch = chunks[i:i + batch_size]

        vector_store.add_documents(batch)

        print(f"Processed {min(i + batch_size, len(chunks))}/{len(chunks)} chunks")

    return vector_store

In [58]:
print("Stored chunks:", vector_store._collection.count())

Stored chunks: 973


In [59]:
results = vector_store.similarity_search(
    "What is machine learning?",
    k=3
)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}:")
    print(doc.page_content[:500])
    print("Metadata:", doc.metadata)


Result 1:
CHAPTER 1
Introduction
Machine learning is about extracting knowledge from data. It is a research field at the
intersection of statistics, artificial intelligence, and computer science and is also
known as predictive analytics or statistical learning. The application of machine
learning methods has in recent years become ubiquitous in everyday life. From auto‐
matic recommendations of which movies to watch, to what food to order or which
products to buy, to personalized online radio and recogniz
Metadata: {'author': 'Andreas C. Müller and Sarah Guido', 'source': 'C:\\Users\\HP\\Downloads\\Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf', 'creationdate': '2016-09-21T13:04:39+00:00', 'keywords': '', 'total_pages': 392, 'file_path': 'C:\\Users\\HP\\Downloads\\Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf', 'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'format': 'PDF 1.6', 'modDate': "D:20200

In [62]:
question = "What is machine learning, and how does it work?"

queries=generate_queries(question)
for i,query in enumerate(queries, start=1):
    print(f"Query {i}: {query}")


all_results = reterving_answers_for_each_multiple_queries(queries)

for i, results in enumerate(all_results, start=1):

    print(f"\nResults for Query {i}:")

    for j, doc in enumerate(results, start=1):

        print(f"\nDocument {j}:")
        print(doc.page_content[:300])

Query 1:  What is the concept of machine learning in artificial intelligence?
Query 2:  How does machine learning differ from traditional data analysis?
Query 3:  What is the process of training a machine to learn from data?
Query 4:  What are the key techniques used in machine learning algorithms?
Query 5:  How does machine learning enable computers to make predictions and decisions?

Results for Query 1:

Document 1:
CHAPTER 1
Introduction
Machine learning is about extracting knowledge from data. It is a research field at the
intersection of statistics, artificial intelligence, and computer science and is also
known as predictive analytics or statistical learning. The application of machine
learning methods has 

Document 2:
Preface
Machine learning is an integral part of many commercial applications and research
projects today, in areas ranging from medical diagnosis and treatment to finding your
friends on social networks. Many people think that machine learning can only be
applied